In [0]:
import json
from pyspark.sql.functions import col, split, element_at, regexp_replace, trim, regexp_extract, to_timestamp, concat_ws, when, from_utc_timestamp, expr, transform, round, explode, slice, size, array_join, get

In [0]:
# Parameters
MAIL = dbutils.widgets.get("MAIL")
USER_NAME = dbutils.widgets.get("USER_NAME")

In [0]:
# --------------
# Files Reading
# --------------
path_all_files = "/Volumes/chess_games/bronze/json/*.json"

df_raw = (spark.read
          .option("multiLine", "true") 
          .json(path_all_files))
          
# 2. Transformer la liste de parties en lignes individuelles
df_game_raws = df_raw.select(explode(col("games")).alias("game_data"))
# 3. Aplatir les colonnes
df_bronze = df_game_raws.select("game_data.*")

df_bronze = df_bronze.filter(
    (col("rated") == True) &
     (col("time_class") == "rapid") &
     (col("time_control") == "600") &
     (col("rules") == "chess"))

In [0]:
df_bronze = df_bronze.drop("end_time", "tcn","fen", "initial_setup", "rules", 'rated', 'time_class', 'time_control', 'match', 'start_time', 'tournament')

In [0]:
# ------
# color
# ------

# Vérifier ma couleur
is_white = col("white").getItem("username") == USER_NAME

df_bronze = df_bronze.withColumns({
    # Mes infos
    "color": when(is_white, "white").otherwise("black"),
    "rating": when(is_white, col("white.rating")).otherwise(col("black.rating")).cast("int"),
    "result_statement": when(is_white, col("white.result")).otherwise(col("black.result")),
    "my_accuracy": when(is_white, col("accuracies.white")).otherwise(col("accuracies.black")),
    
    # Infos adversaire
    "opponent": when(is_white, col("black.username")).otherwise(col("white.username")),
    "opponent_rating": when(is_white, col("black.rating")).otherwise(col("white.rating")).cast("int"),
    "opponent_accuracy": when(is_white, col("accuracies.black")).otherwise(col("accuracies.white"))
})

df_bronze = df_bronze.drop("white", "black", "accuracies")

In [0]:
# -----------------
# result statement
# -----------------

loss_statements = ['resigned', 'checkmated', 'abandoned', 'timeout']
draw_statements = ['agreed', 'stalemate', 'repetition', 'insufficient', 'timevsinsufficient']

df_bronze = df_bronze.withColumn('result', 
    when(col("result_statement").isin(loss_statements), "loss") \
    .when(col("result_statement").isin(draw_statements), "draw") \
    .when(col("result_statement")== 'win', "win") \
    .otherwise("unknown"))


In [0]:
# ----------------------
# Complete opening name
# ----------------------

df_bronze = df_bronze.withColumn(
    "opening_full",  
    element_at(split(col("eco"), "/"), -1))

df_bronze = df_bronze.drop("eco")

In [0]:
# ---------------
# Opening groups
# ---------------

# 1. On split l'ouverture
split_col = split(col("opening_full"), "-")

# 2. On définit la condition pour le 3ème élément
# Il existe ET il ne commence pas par un chiffre (regex ^[0-9])
third_item_exists_and_is_text = (size(split_col) >= 3) & (~split_col[2].rlike("^[0-9]"))

df_bronze = df_bronze.withColumn(
    "opening_group",
    when(third_item_exists_and_is_text, 
         array_join(slice(split_col, 1, 3), "-") # On prend les 3 premiers
    ).otherwise(
         array_join(slice(split_col, 1, 2), "-") # Sinon on n'en prend que 2
    )
)


In [0]:
# ------
# Dates
# ------

# Récupérer  les infos du string pgn
tags = ["UTCDate", "UTCTime"]

# extraction par tag
for tag in tags:
    pattern = rf'\[{tag} \"(.*?)\"\]'
    df_bronze = df_bronze.withColumn(tag, regexp_extract(col("pgn"), pattern, 1))


# format date GMT+1
df_bronze = df_bronze \
    .withColumn("real_timestamp", 
    from_utc_timestamp(
        to_timestamp(concat_ws(" ", col("UTCDate"), col("UTCTime")), "yyyy.MM.dd HH:mm:ss")
        , "Europe/Brussels"))

In [0]:
# ------
# Moves
# ------

# Le regex après \n\n
df_bronze = df_bronze.withColumn("raw_moves", 
    regexp_extract(col("pgn"), r'\n\n(.*)', 1))

In [0]:
# nettoyer les moves
df_bronze = df_bronze.withColumn("cleaned_pgn",
    regexp_replace(col("raw_moves"), 
                   r"(\d+\.+ )|\[%clk |\{\s*|\s*\}|\]", ""))

In [0]:
# transformation en liste
df_bronze = df_bronze.withColumn("move_array", split(trim(col("cleaned_pgn")), " "))
df_bronze = df_bronze.withColumns({
    "white_moves": expr("filter(move_array, (x, i) -> i % 4 == 0)"),
    "white_chr_str": expr("filter(move_array, (x, i) -> i % 4 == 1)"),
    "black_moves": expr("filter(move_array, (x, i) -> i % 4 == 2)"),
    "black_chr_str": expr("filter(move_array, (x, i) -> i % 4 == 3)")
})

In [0]:
df_bronze = df_bronze.withColumn('first_moves',
    concat_ws(' ', 
              get(col('white_moves'), 0), get(col('black_moves'), 0), 
              get(col('white_moves'), 1), get(col('black_moves'), 1), 
              get(col('white_moves'), 2), get(col('black_moves'), 2)))


In [0]:
# ---------------
# Time per moves
# ---------------

# Time : str -> sec
df_bronze = df_bronze.withColumn("white_chr", transform(
    col("white_chr_str"),
    lambda x : (
        (split(x, ":")[0].cast("double") * 3600) +
        (split(x, ":")[1].cast("double") * 60) +
        (split(x, ":")[2].cast("double"))
    ))) 


df_bronze = df_bronze.withColumn("black_chr", transform(
    col("black_chr_str"),
    lambda x : (
        (split(x, ":")[0].cast("double") * 3600) +
        (split(x, ":")[1].cast("double") * 60) +
        (split(x, ":")[2].cast("double"))
    ))) 

In [0]:
# Temps par coup
START_TIME = 600.0

# calcul du temps par coup
df_bronze = df_bronze.withColumn("white_times", transform(
    col("white_chr"),
    lambda x, i: round(when(i == 0, START_TIME - x)
                  .otherwise(col("white_chr")[i-1] - x), 1 )))

df_bronze = df_bronze.withColumn("black_times", transform(
    col("black_chr"),
    lambda x, i: round(when(i == 0, START_TIME - x)
                  .otherwise(col("black_chr")[i-1] - x), 1 )))

In [0]:
# ----------------
# Table insertion
# ----------------

df_bronze = df_bronze.drop("pgn", "UTCDate", "UTCTime", "raw_moves", "cleaned_pgn", "move_array", "white_chr_sec", "black_chr_sec", "black_chr_str", "white_chr_str")

In [0]:
df_bronze.write.format("delta").mode("overwrite").saveAsTable("chess_games.silver.games")